# Model Training Session

This notebook demonstrates how to configure and run an end-to-end continuous training session.

## Environment Setup & Imports

If you are running this notebook on **Databricks**, uncomment and run the following line in the cell below to install the local `landseg` package in your cluster's Python environment:
```python
# %pip install -e ..
```

In [ ]:
# Databricks Setup

import os
import sys
sys.path.insert(0, os.path.join(os.path.abspath('..'), 'src'))
import landseg.adapters.api as api

# Check if running in a Databricks environment
IS_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ

# Define the experiment root path where artifacts and results will be stored.
# Relative paths are standard for local repositories. On Databricks, consider utilizing
# absolute paths on a Unity Catalog Volume or DBFS path (e.g. '/Volumes/my_catalog/my_schema/my_volume/experiment/').
EXP_ROOT = '../experiment/'
# Define name of the dataset, used for naming outputs and logging
DATASET_NAME = 'demo_data'
#
if IS_DATABRICKS:
    # EXP_ROOT = '/Volumes/my_catalog/my_schema/my_volume/experiment/'
    print("Running on Databricks. For high performance, configure EXP_ROOT using a Unity Catalog Volume or DBFS path.")

## Basic Training Configurations (a continuous training session)

In [ ]:
# NOTE
# This is a demo for configuring a simple continuous training session

# init configurator
configurator = api.TrainingSessionConfigurator(EXP_ROOT, DATASET_NAME)

# set data loading parameters
configurator.set_data_loading(
    # number of samples per batch for training and validation dataloaders
    batch_size=16,
    # patch size in pixels (must be able to divide the tile size, e.g., 256/128=2)
    patch_size=128,
)

# set domain source for conditioning
# currently support one categorical and one continuous domain; set to None if not to be used
configurator.set_domain_source(
    # domain used as categorical IDs, name must match the source raster
    category_domain=None,
    # domain used as continuous vectors, name must match the source raster
    continuous_domain=None
)

# set model architecture
configurator.set_model(
    # unet | unetpp | unetppp
    body='unet',
    # cov | transformer | hybrid
    bottleneck='conv',
    # higher value = higher model capacity and computational cost
    base_channel=16,
)

# set optimization
configurator.set_optimization(
    # AdamW |
    optimizer='AdamW',
    learning_rate=1e-4,
    weight_decay=1e-3,
    # CosAnneal |
    scheduler='CosAnneal',
)

# set engine tasks related knobs
configurator.set_tasks(
    # higher value means more adjustment for class imbalance; set to 0 to disable
    logit_adjust_alpha=0.0,
    # {head_name: [cls1, ...]} if None all classes are trained
    exclude_classes=None,
    # relative weights of the losses
    # higher value means more importance in the overall loss and optimization
    loss_weights={
        # major: classification
        'focal': 0.5,
        # major: segmentation quality
        'dice': 0.5,
        # aux spectral consistency
        'spectral': 1e-3,
        # for edge smoothness
        'total_var': 1e-4,
    }
)

# set runtime parameters
configurator.set_runtime(
    # number of epochs to train
    max_epochs=5,
    #[head_name, ...] if None all heads will be trained
    active_heads=None,
    # number of epochs to wait for early stopping; set to None to disable
    patience_epoch=None,
    # {head_name: head_weight, ...} if None track metrics from the first head
    track_heads=None,
)

# run the training session
api.run(configurator.running_root_config)